# Federal Funds Target Rate - FOMC Meeting Data Processing

**Goal:**
1. Process the `Federal Funds Target Range Upper Limit.csv` data.
2. Incorporate explicit FOMC meeting dates to identify decision days.
3. Create labels (`higher`, `same`, `lower`) reflecting the decision made at each meeting compared to the previous meeting.

Source: https://fred.stlouisfed.org/series/FEDFUNDS

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the continuous daily data
file_path = "https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/Federal%20Funds%20Target%20Range%20Upper%20Limit.csv"
df = pd.read_csv(file_path)

# Clean Data
df['DFEDTARU'] = pd.to_numeric(df['DFEDTARU'], errors='coerce')
df['observation_date'] = pd.to_datetime(df['observation_date'])
df = df.dropna(subset=['DFEDTARU']).sort_values('observation_date').reset_index(drop=True)

df.head()

### Incorporate FOMC Meeting Dates

We use the provided list of past FOMC meeting dates. The new target rate takes effect on the meeting day itself or the day after. Because the FRED series is continuous, measuring the target rate exactly **1 day after** the meeting will securely yield the post-decision target rate for that meeting. 

Source: https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm

In [2]:
raw_meetings = [
    '2009-01-28', '2009-03-18', '2009-04-29', '2009-06-24', '2009-08-12', '2009-09-23', '2009-11-04', '2009-12-16', 
    '2010-01-27', '2010-03-16', '2010-04-28', '2010-06-23', '2010-08-10', '2010-09-21', '2010-11-03', '2010-12-14', 
    '2011-01-26', '2011-03-15', '2011-04-27', '2011-06-22', '2011-08-09', '2011-09-21', '2011-11-02', '2011-12-13', 
    '2012-01-25', '2012-03-13', '2012-04-25', '2012-06-20', '2012-07-31', '2012-09-13', '2012-10-24', '2012-12-12', 
    '2013-01-30', '2013-03-20', '2013-05-01', '2013-06-19', '2013-07-31', '2013-09-18', '2013-10-30', '2013-12-18', 
    '2014-01-29', '2014-03-19', '2014-04-30', '2014-06-18', '2014-07-30', '2014-09-17', '2014-10-29', '2014-12-17', 
    '2015-01-28', '2015-03-18', '2015-04-29', '2015-06-17', '2015-07-29', '2015-09-17', '2015-10-28', '2015-12-16', 
    '2016-01-27', '2016-03-16', '2016-04-27', '2016-06-15', '2016-07-27', '2016-09-21', '2016-11-02', '2016-12-14', 
    '2017-02-01', '2017-03-15', '2017-05-03', '2017-06-14', '2017-07-26', '2017-09-20', '2017-11-01', '2017-12-13', 
    '2018-01-31', '2018-03-21', '2018-05-02', '2018-06-13', '2018-08-01', '2018-09-26', '2018-11-08', '2018-12-19', 
    '2019-01-30', '2019-03-19', '2019-05-01', '2019-06-19', '2019-07-31', '2019-09-18', '2019-10-04', '2019-10-30', '2019-12-11', 
    '2020-01-29', '2020-03-03', '2020-03-15', '2020-03-23', '2020-04-29', '2020-06-10', '2020-07-29', '2020-09-16', '2020-11-05', '2020-12-16', 
    '2021-01-27', '2021-03-17', '2021-04-28', '2021-06-16', '2021-07-28', '2021-09-22', '2021-11-03', '2021-12-15', 
    '2022-01-26', '2022-03-16', '2022-05-04', '2022-06-15', '2022-07-27', '2022-09-21', '2022-11-02', '2022-12-14', 
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-14', '2023-07-26', '2023-09-20', '2023-11-01', '2023-12-13', 
    '2024-01-31', '2024-03-20', '2024-05-01', '2024-06-12', '2024-07-31', '2024-09-18', '2024-11-07', '2024-12-18', 
    '2025-01-28', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30', '2025-09-17', '2025-10-29', '2025-12-10', 
    '2026-01-28'
]

# Ensure date formats are correct and drop any accidental duplicates (like 2015-04-29)
df_meetings = pd.DataFrame({'meeting_date': pd.to_datetime(raw_meetings)})
df_meetings = df_meetings.drop_duplicates().sort_values('meeting_date').reset_index(drop=True)

# The decision establishes the target rate that holds the day after the meeting
df_meetings['post_meeting_date'] = df_meetings['meeting_date'] + pd.Timedelta(days=1)

# Merge with the continuous target rate dataframe
df_decisions = pd.merge(
    df_meetings,
    df[['observation_date', 'DFEDTARU']],
    left_on='post_meeting_date',
    right_on='observation_date',
    how='left'
)

# Clean up columns and keep only relevant variables
df_decisions = df_decisions[['meeting_date', 'DFEDTARU']].rename(columns={'DFEDTARU': 'target_rate'})

print("Total meetings matched to rates:", len(df_decisions))
df_decisions.head()

Total meetings matched to rates: 140


,meeting_date,target_rate
0,2009-01-28,0.25
1,2009-03-18,0.25
2,2009-04-29,0.25
3,2009-06-24,0.25
4,2009-08-12,0.25


### Determine FOMC Action Labels (`higher`, `same`, `lower`)

Now we calculate the difference between the target rate produced by the *current* meeting and the target rate maintained from the *previous* meeting.

In [3]:
def get_action_label(diff_val):
    if pd.isna(diff_val):
        return None
    elif diff_val > 0:
        return 'higher'
    elif diff_val < 0:
        return 'lower'
    else:
        return 'same'

# Calculate diff from the immediately preceding meeting
df_decisions['previous_rate'] = df_decisions['target_rate'].shift(1)
df_decisions['rate_change'] = df_decisions['target_rate'] - df_decisions['previous_rate']
df_decisions['decision'] = df_decisions['rate_change'].apply(get_action_label)

display(df_decisions.head(15))

,meeting_date,target_rate,previous_rate,rate_change,decision
0,2009-01-28,0.25,NaN,NaN,NaN
1,2009-03-18,0.25,0.25,0.0,same
2,2009-04-29,0.25,0.25,0.0,same
3,2009-06-24,0.25,0.25,0.0,same
4,2009-08-12,0.25,0.25,0.0,same
5,2009-09-23,0.25,0.25,0.0,same
6,2009-11-04,0.25,0.25,0.0,same
7,2009-12-16,0.25,0.25,0.0,same
8,2010-01-27,0.25,0.25,0.0,same
9,2010-03-16,0.25,0.25,0.0,same


### Descriptive Statistics – Federal Funds Rate

Before visualizing, we summarize the distribution and key properties of the target rate and meeting-level decisions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ── 1. Coverage ──────────────────────────────────────────────────────────────
print("=" * 55)
print("  FEDERAL FUNDS RATE  –  DESCRIPTIVE STATISTICS")
print("=" * 55)
print(f"  Date range   : {df_decisions['meeting_date'].min().date()}  →  {df_decisions['meeting_date'].max().date()}")
print(f"  Total meetings: {len(df_decisions)}")
print()

# ── 2. Core summary stats on target_rate ────────────────────────────────────
print("── Target Rate (%) ──────────────────────────────────")
display(df_decisions[['target_rate']].describe().T.round(4))
print()

# ── 3. Decision label distribution ──────────────────────────────────────────
decision_counts = df_decisions['decision'].value_counts()
decision_pct    = (decision_counts / decision_counts.sum() * 100).round(1)
decision_summary = pd.DataFrame({'count': decision_counts, 'pct (%)': decision_pct})
print("── Decision Distribution ────────────────────────────")
display(decision_summary)
print()

# ── 4. Rate-change statistics (hikes & cuts only) ────────────────────────────
changes = df_decisions[df_decisions['decision'].isin(['higher', 'lower'])].copy()
print("── Rate-Change Episodes ─────────────────────────────")
display(changes[['meeting_date', 'target_rate', 'rate_change', 'decision']]
        .rename(columns={'meeting_date': 'Date', 'target_rate': 'New Rate (%)',
                         'rate_change': 'Change (pp)', 'decision': 'Decision'})
        .reset_index(drop=True))
print()
print(f"  Avg hike size : +{changes[changes['decision']=='higher']['rate_change'].mean():.3f} pp")
print(f"  Avg cut size  :  {changes[changes['decision']=='lower']['rate_change'].mean():.3f} pp")
print()

# ── 5. Histogram of target rate + decision bar chart (side by side) ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of target rate levels
axes[0].hist(df_decisions['target_rate'].dropna(), bins=20,
             color='steelblue', edgecolor='white', linewidth=0.6)
axes[0].set_title('Distribution of Fed Funds Target Rate')
axes[0].set_xlabel('Target Rate (%)')
axes[0].set_ylabel('Number of Meetings')
axes[0].grid(True, alpha=0.3)

# Bar chart of decision counts
colors = {'higher': '#e05c5c', 'same': '#7ab3d4', 'lower': '#5cb87a'}
bars = axes[1].bar(decision_counts.index,
                   decision_counts.values,
                   color=[colors.get(d, 'gray') for d in decision_counts.index],
                   edgecolor='white', linewidth=0.6)
for bar, (label, count) in zip(bars, decision_counts.items()):
    pct = decision_pct[label]
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.8,
                 f'{count}\n({pct}%)', ha='center', va='bottom', fontsize=10)
axes[1].set_title('FOMC Decision Distribution')
axes[1].set_xlabel('Decision')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Visualize Federal Funds Rate
Let's plot the target rate over time.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(x='meeting_date', y='target_rate', data=df_decisions, ax=ax, color='steelblue', linewidth=1.5)
ax.scatter(df_decisions['meeting_date'], df_decisions['target_rate'],
           color='red', s=25, zorder=5, label='FOMC Meeting')
ax.set_title('Federal Funds Target Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Target Rate (%)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Explore Rate Policy Distribution

Let's see an overview of what the Federal Reserve actually did across these meetings.

In [5]:
print("Overall Rate Changes at Meetings (Distribution):")
display(df_decisions['decision'].value_counts())

print("\nSample of Meetings where rates CHANGED:")
display(df_decisions[df_decisions['decision'].isin(['higher', 'lower'])].head(10))

Overall Rate Changes at Meetings (Distribution):


decision
same      108
higher     20
lower      11
Name: count, dtype: int64


Sample of Meetings where rates CHANGED:


,meeting_date,target_rate,previous_rate,rate_change,decision
55,2015-12-16,0.50,0.25,0.25,higher
63,2016-12-14,0.75,0.50,0.25,higher
65,2017-03-15,1.00,0.75,0.25,higher
67,2017-06-14,1.25,1.00,0.25,higher
71,2017-12-13,1.50,1.25,0.25,higher
73,2018-03-21,1.75,1.50,0.25,higher
75,2018-06-13,2.00,1.75,0.25,higher
77,2018-09-26,2.25,2.00,0.25,higher
79,2018-12-19,2.50,2.25,0.25,higher
84,2019-07-31,2.25,2.50,-0.25,lower


---
### Incorporating Crude Oil Data
We now connect the crude oil data to the end of this process.

In [ ]:
# Load WTI Crude Oil Data
df_oil = pd.read_csv('https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/WTI%20Crude%20Oil%20Prices.csv')
df_oil['observation_date'] = pd.to_datetime(df_oil.iloc[:,0], errors='coerce')
df_oil['oil_price'] = pd.to_numeric(df_oil.iloc[:, 1], errors='coerce')
df_oil = df_oil.dropna(subset=['observation_date', 'oil_price']).sort_values('observation_date').reset_index(drop=True)

oil_stats = []
for date in df_decisions['meeting_date']:
    start_date = date - pd.Timedelta(days=30)
    history_30d = df_oil[(df_oil['observation_date'] >= start_date) & (df_oil['observation_date'] < date)]
    if not history_30d.empty:
        avg_price = history_30d['oil_price'].mean()
        if history_30d['oil_price'].iloc[0] != 0:
            price_change_pct = (history_30d['oil_price'].iloc[-1] - history_30d['oil_price'].iloc[0]) / history_30d['oil_price'].iloc[0]
        else:
            price_change_pct = np.nan
    else:
        avg_price = np.nan
        price_change_pct = np.nan
    oil_stats.append({'meeting_date': date, 'oil_price_30d_avg': avg_price, 'oil_price_30d_pct_change': price_change_pct})

df_oil_stats = pd.DataFrame(oil_stats)
df_merged = pd.merge(df_decisions, df_oil_stats, on='meeting_date', how='inner').dropna()
display(df_merged.head())

### Visualize Normalized WTI Crude Oil Price

We normalize the WTI price to the [0, 1] range using min-max scaling so it can be compared directly against the normalized Fed Funds Rate on the same axis.

In [ ]:
# --- Chart 1: Normalized WTI Crude Oil Price (Daily) with FOMC Meeting Markers ---
oil_min = df_oil['oil_price'].min()
oil_max = df_oil['oil_price'].max()
df_oil['oil_price_normalized'] = (df_oil['oil_price'] - oil_min) / (oil_max - oil_min)

# For each FOMC meeting, find the closest available WTI price on or before that date
meeting_wti_rows = []
for date in df_decisions['meeting_date']:
    available = df_oil[df_oil['observation_date'] <= date]
    if not available.empty:
        meeting_wti_rows.append({
            'meeting_date': date,
            'oil_price_normalized': available.iloc[-1]['oil_price_normalized']
        })
df_meeting_wti = pd.DataFrame(meeting_wti_rows)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_oil['observation_date'], df_oil['oil_price_normalized'],
        color='darkorange', linewidth=1.2, label='WTI Crude Oil (Normalized)')
ax.scatter(df_meeting_wti['meeting_date'], df_meeting_wti['oil_price_normalized'],
           color='red', s=25, zorder=5, label='FOMC Meeting')
ax.set_title('Normalized WTI Crude Oil Price Over Time\n(Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price (0–1)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Chart 2: Combined Normalized Fed Rate + WTI (at FOMC Meeting Dates) ---
# Normalize target rate
rate_min = df_merged['target_rate'].min()
rate_max = df_merged['target_rate'].max()
df_merged['target_rate_normalized'] = (df_merged['target_rate'] - rate_min) / (rate_max - rate_min)

# Normalize WTI 30-day average (aligned to meeting dates)
oil_avg_min = df_merged['oil_price_30d_avg'].min()
oil_avg_max = df_merged['oil_price_30d_avg'].max()
df_merged['oil_price_30d_normalized'] = (df_merged['oil_price_30d_avg'] - oil_avg_min) / (oil_avg_max - oil_avg_min)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_merged['meeting_date'], df_merged['target_rate_normalized'],
        color='steelblue', linewidth=2, label='Fed Funds Rate (Normalized)')
ax.plot(df_merged['meeting_date'], df_merged['oil_price_30d_normalized'],
        color='darkorange', linewidth=2, alpha=0.85,
        label='WTI Crude Oil – 30d Avg (Normalized)')

# Highlight meetings where the rate actually changed, with ↑ / ↓ annotations
changed = df_merged[df_merged['decision'].isin(['higher', 'lower'])]
ax.scatter(changed['meeting_date'], changed['target_rate_normalized'],
           color='red', s=60, zorder=6, label='Rate Change (FOMC)')

for _, row in changed.iterrows():
    symbol = '↑' if row['decision'] == 'higher' else '↓'
    ax.annotate(symbol,
                xy=(row['meeting_date'], row['target_rate_normalized']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=9, color='red', fontweight='bold')

ax.set_title('Normalized Federal Funds Rate vs. WTI Crude Oil Price\n(at FOMC Meeting Dates, Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Incorporating Consumer Price Index (CPI) Data

We calculate the Year-over-Year (YoY) percentage change for the CPI data: \\((C_t / C_{t-12} - 1) * 100\\).
Then, we will normalize it and plot it alongside the Federal Funds Rate and WTI Crude Oil.

Source: https://fred.stlouisfed.org/series/CPIAUCSL

In [ ]:
# 1. Calculate Year over Year % and plot it
cpi_path = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/CPI%20All%20Urban%20Consumers.csv'
df_cpi = pd.read_csv(cpi_path)
df_cpi['observation_date'] = pd.to_datetime(df_cpi['observation_date'])
df_cpi['CPIAUCSL'] = pd.to_numeric(df_cpi['CPIAUCSL'], errors='coerce')
df_cpi = df_cpi.dropna(subset=['CPIAUCSL']).sort_values('observation_date').reset_index(drop=True)

# YoY % = (C_t / C_{t-12} - 1) * 100
df_cpi['CPI_YoY_pct'] = (df_cpi['CPIAUCSL'] / df_cpi['CPIAUCSL'].shift(12) - 1) * 100
df_cpi = df_cpi.dropna(subset=['CPI_YoY_pct']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_cpi['observation_date'], df_cpi['CPI_YoY_pct'], color='green', linewidth=1.5, label='CPI YoY %')
ax.set_title('CPI Year-over-Year Percentage Change')
ax.set_xlabel('Date')
ax.set_ylabel('YoY %')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2. Normalize it and plot it
cpi_yoy_min = df_cpi['CPI_YoY_pct'].min()
cpi_yoy_max = df_cpi['CPI_YoY_pct'].max()
df_cpi['CPI_YoY_normalized'] = (df_cpi['CPI_YoY_pct'] - cpi_yoy_min) / (cpi_yoy_max - cpi_yoy_min)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_cpi['observation_date'], df_cpi['CPI_YoY_normalized'], color='purple', linewidth=1.5, label='Normalized CPI YoY %')
ax.set_title('Normalized CPI YoY % (Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Match CPI data to FOMC meeting dates
meeting_cpi_rows = []
for date in df_merged['meeting_date']:
    # Available CPI data on or before the meeting date
    available_cpi = df_cpi[df_cpi['observation_date'] <= date]
    if not available_cpi.empty:
        meeting_cpi_rows.append({
            'meeting_date': date,
            'CPI_YoY_normalized': available_cpi.iloc[-1]['CPI_YoY_normalized']
        })
    else:
        meeting_cpi_rows.append({
            'meeting_date': date,
            'CPI_YoY_normalized': np.nan
        })

df_meeting_cpi = pd.DataFrame(meeting_cpi_rows)
df_merged_all = pd.merge(df_merged, df_meeting_cpi, on='meeting_date', how='left')

# 3. Plot normalized fed rate, WTI, and CPI YoY on the same chart
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_merged_all['meeting_date'], df_merged_all['target_rate_normalized'],
        color='steelblue', linewidth=2, label='Fed Funds Rate (Normalized)')
ax.plot(df_merged_all['meeting_date'], df_merged_all['oil_price_30d_normalized'],
        color='darkorange', linewidth=2, alpha=0.85, label='WTI Crude Oil – 30d Avg (Normalized)')
ax.plot(df_merged_all['meeting_date'], df_merged_all['CPI_YoY_normalized'],
        color='purple', linewidth=2, alpha=0.85, label='CPI YoY % (Normalized)')

changed = df_merged_all[df_merged_all['decision'].isin(['higher', 'lower'])]
ax.scatter(changed['meeting_date'], changed['target_rate_normalized'],
           color='red', s=60, zorder=6, label='Rate Change (FOMC)')

for _, row in changed.iterrows():
    symbol = '↑' if row['decision'] == 'higher' else '↓'
    if not pd.isna(row['target_rate_normalized']):
        ax.annotate(symbol,
                    xy=(row['meeting_date'], row['target_rate_normalized']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=9, color='red', fontweight='bold')

ax.set_title('Normalized Fed Funds Rate vs. WTI Crude Oil vs. CPI YoY\n(at FOMC Meeting Dates, Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Incorporating Unemployment Rate Data

Next, we load the Unemployment Rate, normalize it, and compare it with the Federal Funds Rate, WTI Crude Oil, and CPI YoY data.

In [ ]:
# 1. Load Unemployment Rate, Normalize it, and Plot it
unrate_path = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/Unemployment%20Rate%20UNRATE.csv'
df_unrate = pd.read_csv(unrate_path)
df_unrate['observation_date'] = pd.to_datetime(df_unrate['observation_date'])
df_unrate['UNRATE'] = pd.to_numeric(df_unrate['UNRATE'], errors='coerce')
df_unrate = df_unrate.dropna(subset=['UNRATE']).sort_values('observation_date').reset_index(drop=True)

# Normalize Unemployment Rate
unrate_min = df_unrate['UNRATE'].min()
unrate_max = df_unrate['UNRATE'].max()
df_unrate['UNRATE_normalized'] = (df_unrate['UNRATE'] - unrate_min) / (unrate_max - unrate_min)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_unrate['observation_date'], df_unrate['UNRATE'], color='brown', linewidth=1.5, label='Unemployment Rate (%)')
ax.set_title('Unemployment Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Unemployment Rate (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_unrate['observation_date'], df_unrate['UNRATE_normalized'], color='brown', linewidth=1.5, label='Normalized Unemployment Rate')
ax.set_title('Normalized Unemployment Rate (Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Match Unemployment Rate data to FOMC meeting dates
meeting_unrate_rows = []
for date in df_merged_all['meeting_date']:
    # Available UNRATE data on or before the meeting date
    available_unrate = df_unrate[df_unrate['observation_date'] <= date]
    if not available_unrate.empty:
        meeting_unrate_rows.append({
            'meeting_date': date,
            'UNRATE_normalized': available_unrate.iloc[-1]['UNRATE_normalized']
        })
    else:
        meeting_unrate_rows.append({
            'meeting_date': date,
            'UNRATE_normalized': np.nan
        })

df_meeting_unrate = pd.DataFrame(meeting_unrate_rows)
df_merged_final = pd.merge(df_merged_all, df_meeting_unrate, on='meeting_date', how='left')

# 2. Plot normalized Fed Rate, WTI, CPI YoY, and Unemployment Rate on the same chart
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(df_merged_final['meeting_date'], df_merged_final['target_rate_normalized'],
        color='steelblue', linewidth=2, label='Fed Funds Rate (Normalized)')
ax.plot(df_merged_final['meeting_date'], df_merged_final['oil_price_30d_normalized'],
        color='darkorange', linewidth=2, alpha=0.85, label='WTI Crude Oil – 30d Avg (Normalized)')
ax.plot(df_merged_final['meeting_date'], df_merged_final['CPI_YoY_normalized'],
        color='purple', linewidth=2, alpha=0.85, label='CPI YoY % (Normalized)')
ax.plot(df_merged_final['meeting_date'], df_merged_final['UNRATE_normalized'],
        color='brown', linewidth=2, alpha=0.85, label='Unemployment Rate (Normalized)')

changed = df_merged_final[df_merged_final['decision'].isin(['higher', 'lower'])]
ax.scatter(changed['meeting_date'], changed['target_rate_normalized'],
           color='red', s=60, zorder=6, label='Rate Change (FOMC)')

for _, row in changed.iterrows():
    symbol = '↑' if row['decision'] == 'higher' else '↓'
    if not pd.isna(row['target_rate_normalized']):
        ax.annotate(symbol,
                    xy=(row['meeting_date'], row['target_rate_normalized']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=9, color='red', fontweight='bold')

ax.set_title('Normalized Fed Funds Rate vs. WTI Crude Oil vs. CPI YoY vs. Unemployment Rate\n(at FOMC Meeting Dates, Min-Max Scaled to [0, 1])')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Value (0–1)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part A — ACF Analysis (Diagnosing Lag Structure)

We create a numeric encoding for the FOMC decision (`lower = -1, same = 0, higher = 1`) and run an Autocorrelation Function (ACF) plot on both the `target_rate` and the `decision_num`. 

**What ACF measures:**
The Autocorrelation Function measures how correlated a series is with its own past values at different lags. A lag of 1 means comparing the current meeting's value to the previous meeting's value.

**Interpreting the plots:**
- Spikes that extend beyond the shaded blue region (confidence interval) indicate statistically significant autocorrelation at those lags.
- For `target_rate`, we typically see a slow decay (high inertia), meaning the rate level is highly dependent on recent past levels.
- For `decision_num` (rate changes), the ACF shows if hikes/cuts tend to cluster together or if they revert quickly.

**How we will use this:**
If lags 1 through 3 or 4 are significant, we will construct features using those specific past values to capture the short-term policy trajectory. We will set `MAX_LAG = 4` as a starting point to capture about half a year of meeting history.

In [ ]:
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf

# 1. Create numeric encoding for decision
decision_map = {'lower': -1, 'same': 0, 'higher': 1}
df_merged_final['decision_num'] = df_merged_final['decision'].map(decision_map)

# 2. Plot ACF for target_rate and decision_num
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(df_merged_final['target_rate'].dropna(), lags=12, ax=axes[0],
         title='ACF of Target Rate (Levels)')
axes[0].set_xlabel('Lag (meetings)')
axes[0].set_ylabel('Autocorrelation')

plot_acf(df_merged_final['decision_num'].dropna(), lags=12, ax=axes[1],
         title='ACF of Decision (-1, 0, 1)')
axes[1].set_xlabel('Lag (meetings)')
axes[1].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

## Part B — Feature Engineering (Lags, Trends, Policy Inertia)

Here we build a rich set of features to predict the meeting's policy action. All features refer strictly to information from **prior** meetings to prevent data leakage.

In [ ]:
# Identify core predictor columns available in df_merged_final
target_col = 'target_rate'
cpi_col = 'CPI_YoY_normalized' if 'CPI_YoY_normalized' in df_merged_final.columns else 'CPI_YoY_pct'
unemp_col = 'UNRATE_normalized' if 'UNRATE_normalized' in df_merged_final.columns else 'UNRATE'
wti_col = 'oil_price_30d_avg' 
wti_col_norm = 'oil_price_30d_normalized'

# Let's create a fresh dataframe for modeling to keep things clean
df_model = df_merged_final[['meeting_date', 'decision', 'decision_num', target_col, cpi_col, unemp_col, wti_col]].copy()

MAX_LAG = 4

# 4. Create lag features (meeting frequency)
lag_cols = [target_col, cpi_col, unemp_col, wti_col]
for col in lag_cols:
    for lag in range(1, MAX_LAG + 1):
        df_model[f'{col}_lag{lag}'] = df_model[col].shift(lag)

# 5. Create "trend/acceleration" features (based strictly on lags to avoid leakage directly)
# Note: A meeting-to-meeting change from t-2 to t-1 is known at time t.
df_model['CPI_change_m2m'] = df_model[f'{cpi_col}_lag1'] - df_model[f'{cpi_col}_lag2']
df_model['UNRATE_change_2m'] = df_model[f'{unemp_col}_lag1'] - df_model[f'{unemp_col}_lag3']
df_model['WTI_return_meeting'] = (df_model[f'{wti_col}_lag1'] - df_model[f'{wti_col}_lag2']) / (df_model[f'{wti_col}_lag2'] + 1e-6)

df_model.head()

In [ ]:
# 6. Create "policy inertia / duration" features
df_model['prev_decision'] = df_model['decision'].shift(1)
df_model['prev_decision_num'] = df_model['decision_num'].shift(1)

# Tracking time since last non-zero rate change
time_since_last_change = []
consecutive_same_list = []

steps_since_change = 0
consecutive_same = 0

for i in range(len(df_model)):
    # For the current meeting 'i', we record the trailing states computed UP TO meeting 'i-1'
    time_since_last_change.append(steps_since_change)
    consecutive_same_list.append(consecutive_same)
    
    # Now process meeting 'i' to update the state for the next meeting (i+1)
    curr_decision = df_model.loc[i, 'decision_num']
    if not pd.isna(curr_decision):
        if curr_decision != 0:
            steps_since_change = 0
            consecutive_same = 0
        else:
            steps_since_change += 1
            consecutive_same += 1
            
df_model['time_since_last_change'] = time_since_last_change
df_model['consecutive_same'] = consecutive_same_list

# We don't want to use current meeting features as inputs (to prevent leakage)
# So we drop the non-lagged continuous features. But we keep decision/decision_num as our targets.
drop_cols = [target_col, cpi_col, unemp_col, wti_col]
df_model.drop(columns=drop_cols, inplace=True)

# 8. Drop NaNs caused by the required shift lags (MAX_LAG=4 means first 4 rows have NaNs)
df_model = df_model.dropna().reset_index(drop=True)

print("Shape of final modeling DataFrame:", df_model.shape)
display(df_model.head())
display(df_model.describe())

## Part C — Simple Baseline Setup (Walk-Forward Validation)

We construct two naive baselines evaluated with an expanding window (time-series split):
1. **Majority Class Baseline:** Predicts the most frequent decision up to time $t-1$.
2. **Last Decision Baseline:** Predicts that the decision at time $t$ will be whatever the decision was at time $t-1$.

We report overall accuracy and print a confusion matrix for the predictions.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

# Initialize minimum training size to evaluate
INITIAL_TRAIN_SIZE = 40  # Start testing after ~5 years of meetings
target_array = df_model['decision_num'].values

preds_majority = []
preds_last = []
actuals = []

# Walk-forward expanding window
for t in range(INITIAL_TRAIN_SIZE, len(df_model)):
    train_history = target_array[:t]
    actual_t = target_array[t]
    
    # 1. Baseline Majority: most common element in historical window
    majority_class = pd.Series(train_history).mode()[0]
    
    # 2. Baseline Last: previous meeting's decision
    last_class = train_history[-1]
    
    preds_majority.append(majority_class)
    preds_last.append(last_class)
    actuals.append(actual_t)

# Output Accuracy
acc_majority = accuracy_score(actuals, preds_majority)
acc_last = accuracy_score(actuals, preds_last)

print(f"Evaluated on {len(actuals)} out-of-sample forward meetings.")
print(f"Majority Class Baseline Accuracy: {acc_majority:.4f}")
print(f"Last Decision Baseline Accuracy:  {acc_last:.4f}")

# Confusion Matrix for the Last Baseline
labels = [-1, 0, 1]  # lower, same, higher
cm = confusion_matrix(actuals, preds_last, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Lower (-1)', 'Same (0)', 'Higher (1)'],
            yticklabels=['Lower (-1)', 'Same (0)', 'Higher (1)'], ax=ax)
ax.set_title('Confusion Matrix: Last Decision Baseline')
ax.set_xlabel('Predicted Action')
ax.set_ylabel('Actual Action')
plt.tight_layout()
plt.show()

In [ ]:
import os

# Export df_model to disk (optional — skipped gracefully if directory doesn't exist)
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)
output_parquet = os.path.join(output_dir, 'df_model.parquet')
output_csv     = os.path.join(output_dir, 'df_model.csv')

try:
    df_model.to_parquet(output_parquet, index=False)
    print("Saved parquet to:", output_parquet)
except Exception as e:
    print(f"Could not save parquet (possibly missing pyarrow/fastparquet): {e}")

try:
    df_model.to_csv(output_csv, index=False)
    print("Saved CSV to    :", output_csv)
except Exception as e:
    print(f"Could not save CSV: {e}")

print("Shape   :", df_model.shape)
print("Columns :", df_model.columns.tolist())